# exp3 — EquiformerV2 with vs without auxiliary tasks, scaffold split (Colab GPU)
**Status: not run.** Needs an approved compute budget. See `config.yaml`. GPLv3 upstream: minotm/active-drug-discovery.

In [ ]:
!nvidia-smi
!git clone https://github.com/minotm/active-drug-discovery && cd active-drug-discovery && git checkout c687cb1c52704a249f929c81bff5ca0ba21020f6
%cd active-drug-discovery
!pip -q install rdkit torch_geometric torchmetrics e3nn timm mendeleev openpyxl pyyaml hyperopt
# torch_scatter/torch_cluster wheels for the Colab torch version:
import torch; v=torch.__version__.split('+')[0]; cu='cu'+torch.version.cuda.replace('.','')
!pip -q install torch_scatter torch_cluster torch_sparse pyg_lib -f https://data.pyg.org/whl/torch-{v}+{cu}.html

In [ ]:
# Scaffold split (deterministic)
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
df=pd.read_csv('data/training_data/regioselectivity.csv',sep='\t')
df['scaf']=df.SMILES.map(lambda s: MurckoScaffold.MurckoScaffoldSmiles(smiles=s))
groups=sorted(df.groupby('scaf').groups.values(), key=len, reverse=True)
train_idx,test_idx=[],[]
for g in groups:
    (train_idx if len(train_idx)+len(g) <= 0.8*len(df) else test_idx).extend(g)
df.loc[train_idx].drop(columns='scaf').to_csv('regio_scaf_train.csv',sep='\t',index=False)
te=df.loc[test_idx].drop(columns='scaf').rename(columns={'SMILES':'SMILES','product_smiles':'product_smiles'})
te['ID']=range(len(te)); te.to_csv('regio_scaf_test.csv',index=False)
print(len(train_idx),len(test_idx), df.loc[train_idx,'scaf'].isin(df.loc[test_idx,'scaf']).sum())

In [ ]:
# Timing probe: 2 epochs per arm -> extrapolate budget BEFORE the full run
import time,subprocess
for arm,flags in [('no_aux',''),('with_aux','--coordinate_denoising --node_masking')]:
    t=time.time()
    subprocess.run(f'python train_gnn_workflow.py --input_csv regio_scaf_train.csv --epochs 2 --patience 2 --seed 0 --checkpoint_path ckpt/probe_{arm}/ {flags}',shell=True,check=True)
    print(arm,'s/epoch ~',(time.time()-t)/2)

In [ ]:
# Full run (only after the budget is approved)
import yaml,subprocess
cfg=yaml.safe_load(open('/content/config.yaml')) if False else None  # upload config.yaml or edit below
ARMS={'no_aux':'','with_aux':'--coordinate_denoising --node_masking'}
for seed in [0,1,2]:
    for arm,flags in ARMS.items():
        subprocess.run(f'python train_gnn_workflow.py --input_csv regio_scaf_train.csv --epochs 300 --patience 50 --seed {seed} --run_id {seed} --checkpoint_path ckpt/{arm}_s{seed}/ {flags}',shell=True,check=True)

In [ ]:
# Evaluate on scaffold test with apps/active-borylation/src/regio_json.py (copy it into this folder)
# python regio_json.py regio_scaf_test.csv res_{arm}_s{seed}.json --seed {seed} --smiles_col SMILES --id_col ID --checkpoint_folder ckpt/{arm}_s{seed}/
# then top1_site_accuracy = mean(top1_correct); write results.csv / results.md / plot.png, breakpoints.csv for the extreme ladder.